# §17 — Paramétrisation du profil ρ₀⟨u′w′⟩(z)

Objectif : ajuster une forme analytique simple sur le profil de flux de Reynolds calculé en §16,  
afin de produire une formule utilisable dans un GCM.

**Prérequis** : avoir exécuté §16 → `flux_glob_h`, `flux_glob_s`, `flux_loc_h`, `flux_loc_s` en mémoire.

---
## Étape 1 — Visualisation du profil de référence

Avant tout fit, on observe le profil pour choisir la bonne forme analytique :
- est-il nul en surface ?
- est-il nul au sommet de la convection (~15 km) ?
- a-t-il un seul extremum ou plusieurs ?
- change-t-il de signe ?

In [ ]:
# ======================================================================
# §17 — Étape 1 : visualisation du profil flux_glob_h / flux_glob_s
# ======================================================================

idx_tropo = np.searchsorted(alt, 15000)
sc = 1e3   # kg/m²/s² → ×10⁻³

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    ax.plot(flux_glob_h[sl] * sc, alt[sl],
            color='royalblue',   lw=2.5, label='humide (global)')
    ax.plot(flux_glob_s[sl] * sc, alt[sl],
            color='saddlebrown', lw=2.5, label='sèche  (global)')
    ax.plot(flux_loc_h[sl]  * sc, alt[sl],
            color='royalblue',   lw=1.5, linestyle='--', label='humide (local)')
    ax.plot(flux_loc_s[sl]  * sc, alt[sl],
            color='saddlebrown', lw=1.5, linestyle='--', label='sèche  (local)')
    ax.axvline(0, color='grey', alpha=0.5, lw=1)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle(
    '§17 — Étape 1 : profil ρ₀⟨u\'w\'⟩ par région\n'
    'Observer : zéros, extremum, changement de signe',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

# ── diagnostics numériques ─────────────────────────────────────────────
for label, flux in [('HUMIDE (global)', flux_glob_h), ('SÈCHE  (global)', flux_glob_s)]:
    phi = flux[:idx_tropo]
    iz_min = np.argmin(phi)
    iz_max = np.argmax(phi)
    print(f'── Région {label} ──')
    print(f'  min  : {phi[iz_min]*sc:.4f} ×10⁻³  à z = {alt[iz_min]:.0f} m')
    print(f'  max  : {phi[iz_max]*sc:.4f} ×10⁻³  à z = {alt[iz_max]:.0f} m')
    print(f'  surface (iz=0)   : {flux[0]*sc:.4f} ×10⁻³')
    print(f'  z≈15 km (iz={idx_tropo}) : {flux[idx_tropo]*sc:.4f} ×10⁻³')
    print()

---
## Étape 2 — Choix de la forme analytique

On teste la forme canonique CMT (profil triangulaire asymétrique) :

$$\rho_0\,\overline{u'w'}(z) = A \cdot \frac{z}{z_c} \cdot \left(1 - \frac{z}{z_c}\right)^n, \quad z \leq z_c$$

avec :
- $A$ : amplitude (kg/m²/s²)
- $z_c$ : hauteur du sommet convectif (m)
- $n$ : exposant d'asymétrie (contrôle où se trouve le max)

Ce profil est nul en $z=0$ et en $z=z_c$, positif entre les deux si $A>0$.

In [ ]:
# ======================================================================
# §17 — Étape 2 : définition de la forme analytique
# ======================================================================

from scipy.optimize import curve_fit

def profil_cmt(z, A, zc, n):
    """Profil triangulaire asymétrique CMT.
    Nul en z=0 et z=zc, extremum entre les deux.
    """
    xi  = z / zc
    val = A * xi * (1.0 - xi)**n
    val = np.where(z > zc, 0.0, val)   # force à zéro au-dessus
    return val

# Visualisation de la forme pour différents n (sans fit)
z_test = np.linspace(0, 15000, 300)
zc_test = 12000.0

fig, ax = plt.subplots(figsize=(6, 7))
for n_test in [0.5, 1.0, 1.5, 2.0, 3.0]:
    phi_test = profil_cmt(z_test, A=1.0, zc=zc_test, n=n_test)
    ax.plot(phi_test, z_test, lw=2, label=f'n = {n_test}')
ax.axvline(0, color='grey', alpha=0.4)
ax.set_xlabel('A × f(z/zc)  (normalisé)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Forme analytique CMT pour différents n\n(A=1, zc=12 km)', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Étape 3 — Fit scipy sur la région humide

On ajuste les 3 paramètres $(A, z_c, n)$ sur `flux_glob_h` dans la troposphère.

In [ ]:
# ======================================================================
# §17 — Étape 3 : fit sur la région humide (anomalies globales)
# ======================================================================

# -- données à fitter : troposphère uniquement
z_fit   = alt[:idx_tropo].astype(float)
phi_fit = flux_glob_h[:idx_tropo]

# -- valeurs initiales
p0 = [phi_fit.min(), 12000.0, 1.5]

popt_h, pcov_h = curve_fit(
    profil_cmt, z_fit, phi_fit,
    p0=p0, maxfev=20000
)
A_h, zc_h, n_h = popt_h

print('── Fit région HUMIDE ──')
print(f'  A  = {A_h:.4e} kg/m²/s²')
print(f'  zc = {zc_h/1000:.2f} km')
print(f'  n  = {n_h:.3f}')

---
## Étape 4 — Évaluation de la qualité du fit (R²)

$$R^2 = 1 - \frac{\sum_i (\phi_i - \hat{\phi}_i)^2}{\sum_i (\phi_i - \bar{\phi})^2}$$

R² = 1 → fit parfait. On vise R² > 0.90.

In [ ]:
# ======================================================================
# §17 — Étape 4 : R² et résidus
# ======================================================================

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1.0 - ss_res / ss_tot

# ── humide ────────────────────────────────────────────────────────────
phi_pred_h = profil_cmt(z_fit, *popt_h)
R2_h       = r2_score(phi_fit, phi_pred_h)
residus_h  = phi_fit - phi_pred_h

print(f'R² humide = {R2_h:.4f}')
print(f'Résidu max : {np.abs(residus_h).max()*sc:.4f} ×10⁻³ kg/m²/s²')
print(f'Résidu rms : {np.sqrt((residus_h**2).mean())*sc:.4f} ×10⁻³ kg/m²/s²')

# ── résidus en fonction de z ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 7))
ax.plot(residus_h * sc, z_fit, color='royalblue', lw=2)
ax.axvline(0, color='grey', alpha=0.5)
ax.set_xlabel('Résidu (×10⁻³ kg/m²/s²)')
ax.set_ylabel('Altitude (m)')
ax.set_title(f'§17 — Résidus du fit (région humide)\nR² = {R2_h:.4f}', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Étape 5 — Visualisation : simulation vs fit

In [ ]:
# ======================================================================
# §17 — Étape 5 : visualisation simulation vs fit (humide)
# ======================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    ax.plot(flux_glob_h[sl] * sc, alt[sl],
            color='royalblue', lw=2.5, label='simulation (humide global)')
    # fit étendu sur la colonne complète
    phi_fit_full = profil_cmt(alt[sl].astype(float), *popt_h)
    ax.plot(phi_fit_full * sc, alt[sl],
            color='red', lw=2, linestyle='--',
            label=f'fit  A={A_h:.2e}  zc={zc_h/1e3:.1f}km  n={n_h:.2f}  R²={R2_h:.3f}')
    ax.axvline(0, color='grey', alpha=0.4)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle(
    '§17 — Paramétrisation CMT : simulation vs fit analytique\n'
    r'$\rho_0\overline{u\'w\'}(z) = A\,(z/z_c)\,(1-z/z_c)^n$',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Étape 6 — Fit sur la région sèche

Même procédure sur `flux_glob_s` pour comparer les paramètres entre régions.

In [ ]:
# ======================================================================
# §17 — Étape 6 : fit sur la région sèche
# ======================================================================

phi_fit_s = flux_glob_s[:idx_tropo]
p0_s = [phi_fit_s.min(), 12000.0, 1.5]

popt_s, pcov_s = curve_fit(
    profil_cmt, z_fit, phi_fit_s,
    p0=p0_s, maxfev=20000
)
A_s, zc_s, n_s = popt_s

phi_pred_s = profil_cmt(z_fit, *popt_s)
R2_s = r2_score(phi_fit_s, phi_pred_s)

print('── Fit région SÈCHE ──')
print(f'  A  = {A_s:.4e} kg/m²/s²')
print(f'  zc = {zc_s/1000:.2f} km')
print(f'  n  = {n_s:.3f}')
print(f'  R² = {R2_s:.4f}')

# ── comparaison humide vs sèche ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)

for ax, sl, titre in zip(
    axes,
    [slice(None, idx_tropo), slice(None)],
    ['Troposphère (0–15 km)', 'Colonne complète'],
):
    # humide
    ax.plot(flux_glob_h[sl] * sc, alt[sl],
            color='royalblue', lw=2.5, label='simulation humide')
    ax.plot(profil_cmt(alt[sl].astype(float), *popt_h) * sc, alt[sl],
            color='royalblue', lw=1.8, linestyle='--',
            label=f'fit humide  R²={R2_h:.3f}')
    # sèche
    ax.plot(flux_glob_s[sl] * sc, alt[sl],
            color='saddlebrown', lw=2.5, label='simulation sèche')
    ax.plot(profil_cmt(alt[sl].astype(float), *popt_s) * sc, alt[sl],
            color='saddlebrown', lw=1.8, linestyle='--',
            label=f'fit sèche   R²={R2_s:.3f}')

    ax.axvline(0, color='grey', alpha=0.4)
    ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
    ax.set_title(titre, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Altitude (m)')
plt.suptitle(
    '§17 — Fit analytique : humide vs sèche',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Étape 7 — Tableau de synthèse des paramètres

Résumé des paramètres ajustés pour les deux régions.

In [ ]:
# ======================================================================
# §17 — Étape 7 : tableau de synthèse
# ======================================================================

print('=' * 55)
print(f'{"Paramètre":<20} {"Humide":>15} {"Sèche":>15}')
print('=' * 55)
print(f'{"A (kg/m²/s²)":<20} {A_h:>15.4e} {A_s:>15.4e}')
print(f'{"zc (km)":<20} {zc_h/1000:>15.2f} {zc_s/1000:>15.2f}')
print(f'{"n":<20} {n_h:>15.3f} {n_s:>15.3f}')
print(f'{"R²":<20} {R2_h:>15.4f} {R2_s:>15.4f}')
print('=' * 55)

# altitude du maximum analytique : z_max = zc * 1 / (1 + n)
z_max_h = zc_h / (1 + n_h)
z_max_s = zc_s / (1 + n_s)
print(f'{"z_max théorique (km)":<20} {z_max_h/1000:>15.2f} {z_max_s/1000:>15.2f}')
print()
print('Note : z_max = zc / (1 + n)  (dérivée analytique du profil CMT)')

---
## Étape 8 — Robustesse temporelle

On vérifie si les paramètres sont stables au cours de la simulation  
en découpant l'état stationnaire en deux moitiés et en refittant sur chacune.

In [ ]:
# ======================================================================
# §17 — Étape 8 : robustesse temporelle (split early / late)
# ======================================================================

t_mid = t_stat + (n_t - t_stat) // 2   # milieu de l'état stationnaire

flux_early_h = np.zeros(n_z)
flux_late_h  = np.zeros(n_z)

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))

for iz in range(n_z):

    for flux_arr, t0, t1 in [
        (flux_early_h, t_stat, t_mid),
        (flux_late_h,  t_mid,  n_t),
    ]:
        acc = 0.0
        n   = 0
        for it in range(t0, t1):
            u_2d = ds_u['ua'].isel({dim_t: it, dim_z: iz}).values
            w_2d = ds_w['wa'].isel({dim_t: it, dim_z: iz}).values
            u_p  = u_2d - u_2d.mean()
            w_p  = w_2d - w_2d.mean()
            acc += (u_p * w_p)[mh].mean()
            n   += 1
            del u_2d, w_2d, u_p, w_p
        flux_arr[iz] = rho0[iz] * acc / n

    if iz % 10 == 0:
        print(f'  iz={iz}/{n_z-1}')

ds_u.close() ; ds_w.close()
del ds_u, ds_w
gc.collect()

# ── fit sur chaque moitié ──────────────────────────────────────────────
popt_early, _ = curve_fit(profil_cmt, z_fit, flux_early_h[:idx_tropo], p0=p0, maxfev=20000)
popt_late,  _ = curve_fit(profil_cmt, z_fit, flux_late_h[:idx_tropo],  p0=p0, maxfev=20000)

R2_early = r2_score(flux_early_h[:idx_tropo], profil_cmt(z_fit, *popt_early))
R2_late  = r2_score(flux_late_h[:idx_tropo],  profil_cmt(z_fit, *popt_late))

print()
print('── Robustesse temporelle (région humide) ──')
print(f'{"":<8} {"A (kg/m²/s²)":>16} {"zc (km)":>10} {"n":>8} {"R²":>8}')
print(f'{"early":<8} {popt_early[0]:>16.4e} {popt_early[1]/1e3:>10.2f} {popt_early[2]:>8.3f} {R2_early:>8.4f}')
print(f'{"late":<8} {popt_late[0]:>16.4e}  {popt_late[1]/1e3:>10.2f} {popt_late[2]:>8.3f} {R2_late:>8.4f}')
print(f'{"full":<8} {A_h:>16.4e}  {zc_h/1e3:>10.2f} {n_h:>8.3f} {R2_h:>8.4f}')

# ── visualisation ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 9))

ax.plot(flux_glob_h[:idx_tropo] * sc, z_fit,
        color='royalblue', lw=2.5, label='simulation (full)')
ax.plot(flux_early_h[:idx_tropo] * sc, z_fit,
        color='steelblue', lw=1.5, linestyle='--', label='simulation early')
ax.plot(flux_late_h[:idx_tropo]  * sc, z_fit,
        color='navy',      lw=1.5, linestyle=':',  label='simulation late')
ax.plot(profil_cmt(z_fit, *popt_h)     * sc, z_fit,
        color='red', lw=2, linestyle='-',  label=f'fit full    R²={R2_h:.3f}')
ax.plot(profil_cmt(z_fit, *popt_early) * sc, z_fit,
        color='orange', lw=1.5, linestyle='--', label=f'fit early   R²={R2_early:.3f}')
ax.plot(profil_cmt(z_fit, *popt_late)  * sc, z_fit,
        color='darkorange', lw=1.5, linestyle=':', label=f'fit late    R²={R2_late:.3f}')

ax.axvline(0, color='grey', alpha=0.4)
ax.set_xlabel('ρ₀ ū\'w\'  (×10⁻³ kg/m²/s²)')
ax.set_ylabel('Altitude (m)')
ax.set_title('§17 — Robustesse temporelle du fit (humide)\n'
             'early = 1ère moitié stat / late = 2ème moitié',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()